# Session 1: a cutter follows a path to carve a shape
**A. Real-world problem.** A CNC (computer numerical control) machine moves a spinning
cutter along a list of coordinates. A plotter moves a pen the same way. If you want a square
pocket, where must the *centre* of the cutter go, and how long does it take?

**Plan.** (1) a path is a list of points, (2) the cutter centre stays one radius inside the
wall, (3) one lap leaves an island in the middle, (4) time = path length / feed.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a path is a list of points
A plotter pen goes to (x, y), then (x, y) again, and so on. The path length is the sum of
the straight distances between neighbouring points. All lengths here are in millimetres (mm).

In [ ]:
def polyline_length(points):
    """Total length of the straight pieces joining the points (mm)."""
    pts = np.asarray(points, dtype=float)
    return float(np.sum(np.hypot(*np.diff(pts, axis=0).T)))


def square_outline(side):
    """The wall of a square pocket, corner (0, 0) to (side, side), as a closed list of points."""
    return np.array([[0, 0], [side, 0], [side, side], [0, side], [0, 0]], dtype=float)

## Step 2: the cutter centre stays one radius inside the wall
The cutter has a radius r (half its diameter). If its edge must touch the wall, its centre
must be r away from the wall. So the centre path is the wall moved inwards by r: a smaller
square of side (side - 2r).

In [ ]:
def centre_path(side, r):
    """Closed square path of the cutter centre, or None if the cutter does not fit."""
    if 2 * r > side:
        return None
    return np.array([[r, r], [side - r, r], [side - r, side - r], [r, side - r], [r, r]], dtype=float)

## Step 3: one lap leaves an island
The cutter removes material up to r on each side of its centre path. After one lap the
strip next to the wall is clear, but a square island of side (side - 4r) may remain.

In [ ]:
def distance_to_path(point, path):
    """Shortest distance from a point to a polyline (mm)."""
    p = np.asarray(point, dtype=float)
    best = np.inf
    for a, b in zip(path[:-1], path[1:]):
        ab = b - a
        t = 0.0 if not ab.any() else float(np.clip(np.dot(p - a, ab) / np.dot(ab, ab), 0.0, 1.0))
        best = min(best, float(np.linalg.norm(p - (a + t * ab))))
    return best


def island_side(side, r):
    """Side of the square island left after one lap (0 if none)."""
    return max(0.0, side - 4 * r)


def is_cut(point, path, r):
    """True if the cutter, following the path, touches this point."""
    return distance_to_path(point, path) <= r + 1e-9

## Step 4: how long does it take?
Feed rate F is how fast the cutter moves along the path, in mm per minute. The time for a
path of length L is L / F minutes.

In [ ]:
def lap_time_s(side, r, feed):
    """Time in seconds for one lap of the centre path at the feed (mm/min)."""
    path = centre_path(side, r)
    if path is None:
        return float("nan")
    return polyline_length(path) / feed * 60.0

## Step 5: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    side, d, feed = 20.0, 4.0, 120.0
    r = d / 2
    path = centre_path(side, r)
    print(f"pocket {side:g} mm, cutter diameter {d:g} mm, feed {feed:g} mm/min")
    print(f"centre path length = {polyline_length(path):.1f} mm, time = {lap_time_s(side, r, feed):.1f} s")
    print(f"island side after one lap = {island_side(side, r):.1f} mm")

    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    wall = square_outline(side)
    ax.plot(wall[:, 0], wall[:, 1], "k-", lw=2, label="pocket wall")
    ax.plot(path[:, 0], path[:, 1], "C0--", lw=1.5, label="cutter centre path")
    for corner in path[:4]:
        ax.add_patch(plt.Circle(corner, r, fill=False, color="C1", lw=1))
    ax.add_patch(plt.Circle(path[0], r, color="C1", alpha=0.35, label="cutter (diameter 4 mm)"))
    isl = island_side(side, r)
    c0 = (side - isl) / 2
    ax.add_patch(plt.Rectangle((c0, c0), isl, isl, color="C2", alpha=0.35, label="island left"))
    ax.set_aspect("equal")
    ax.set_xlim(-2, side + 2)
    ax.set_ylim(-2, side + 2)
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title("A square pocket, the centre path and the island")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "pocket_lap.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ds = np.linspace(1, 9.5, 60)
    ax2.plot(ds, [polyline_length(centre_path(side, x / 2)) for x in ds], "C0")
    ax2.set_xlabel("cutter diameter (mm)")
    ax2.set_ylabel("one-lap path length (mm)")
    ax2.set_title("Bigger cutter: shorter lap (20 mm pocket)")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "lap_length_vs_diameter.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use the app to predict the island side for a 30 mm pocket with a 6 mm cutter before you read it off the readout.
2. **Stretch.** Write `ring_path(side, r, k)` that returns the square path of the cutter centre for the k-th lap inwards (each lap 2 mm further in) and check that the first lap equals `centre_path`.
3. **Explain.** Explain why the time to cut a path is halved when the feed is doubled, and why that does not mean a real machine can use any feed.

## Assessment
- **A. Conceptual.** Why does the cutter centre have to follow a path one radius inside the wall, and why does one lap of a big pocket leave an island?
- **B. Numerical.** A 30 mm square pocket is cut with a 6 mm cutter at 150 mm/min. Find the path length, the time for one lap and the island side.
- **C. Graph and simulation interpretation.** In the time graph the curve falls as the feed rises. If the feed is halved from 120 to 60 mm/min for the 64 mm path, what happens to the time? What does the curve shape say?
- **D. Design and optimisation challenge.** Design: choose a cutter diameter and a feed to clear a 20 mm pocket in one lap, with no island, in under 20 s. Is there a limit on the feed?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("a closed square outline of side 20 mm has length 80 mm",
                   abs(L.polyline_length(L.square_outline(20)) - 80.0) < 1e-9))
    path = L.centre_path(20, 2.0)
    checks.append(("worked example: 20 mm pocket, 4 mm cutter: centre path is the square 2..18 mm",
                   np.allclose(path[:4], [[2, 2], [18, 2], [18, 18], [2, 18]])))
    checks.append(("worked example: centre path length is 64 mm", abs(L.polyline_length(path) - 64.0) < 1e-9))
    checks.append(("worked example: at 120 mm/min one lap takes 32.0 s", abs(L.lap_time_s(20, 2.0, 120) - 32.0) < 1e-9))
    checks.append(("worked example: the island left after one lap is 12 mm wide", abs(L.island_side(20, 2.0) - 12.0) < 1e-9))
    checks.append(("every centre-path corner is exactly one radius from the walls",
                   all(min(x, 20 - x, y, 20 - y) == 2.0 for x, y in path[:4])))
    checks.append(("the island centre (10, 10) is NOT touched by the 4 mm cutter",
                   not L.is_cut((10, 10), path, 2.0)))
    checks.append(("a point 1.5 mm from the wall IS cut", L.is_cut((1.5, 10), path, 2.0)))
    big = L.centre_path(20, 5.0)
    checks.append(("edge: a 10 mm cutter in a 20 mm pocket has island 0 and cuts the centre",
                   L.island_side(20, 5.0) == 0.0 and L.is_cut((10, 10), big, 5.0)))
    checks.append(("edge: the 10 mm cutter lap is 40 mm long and takes 12.0 s at 200 mm/min",
                   abs(L.polyline_length(big) - 40.0) < 1e-9 and abs(L.lap_time_s(20, 5.0, 200) - 12.0) < 1e-9))
    checks.append(("edge: a 24 mm cutter does not fit a 20 mm pocket (no path)", L.centre_path(20, 12.0) is None))
    checks.append(("half the feed doubles the time", abs(L.lap_time_s(20, 2.0, 60) - 2 * L.lap_time_s(20, 2.0, 120)) < 1e-9))
    checks.append(("a bigger cutter makes a shorter lap (30 mm pocket: 4 mm 104 mm, 8 mm 88 mm)",
                   abs(L.polyline_length(L.centre_path(30, 2.0)) - 104.0) < 1e-9
                   and abs(L.polyline_length(L.centre_path(30, 4.0)) - 88.0) < 1e-9))
    checks.append(("exercise/assessment: 30 mm pocket, 6 mm cutter, 150 mm/min: path 96 mm, 38.4 s, island 18 mm",
                   abs(L.polyline_length(L.centre_path(30, 3.0)) - 96.0) < 1e-9 and abs(L.lap_time_s(30, 3.0, 150) - 38.4) < 1e-9
                   and abs(L.island_side(30, 3.0) - 18.0) < 1e-9))
    checks.append(("design question: the 40 mm lap takes exactly 20.0 s at 120 mm/min and 64 s for the 64 mm lap at 60 mm/min",
                   abs(L.lap_time_s(20, 5.0, 120) - 20.0) < 1e-9 and abs(L.lap_time_s(20, 2.0, 60) - 64.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
